# Manual Soft Prompt Tuning for Vision Language Models

This notebook implements manual soft prompt tuning for VLMs on the RSICD dataset. Supports easy switching between:

- Qwen 2.5 7B VL
- Gemma3 4B (text-only)
- Pixtral 12B
- Phi 3.5 Vision 4.2B

by [Gayanuka Amarasuriya](https://gayanukaa.github.io/)


In [ ]:
!pip install -q torch transformers datasets accelerate scikit-learn pycocoevalcap

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoProcessor, AutoModel, AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer
import time
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image

# Import pycocoevalcap for evaluation
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

## Configurations


In [ ]:
# Model selection - uncomment one
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
# MODEL_NAME = "microsoft/Phi-3.5-vision-instruct"
# MODEL_NAME = "mistralai/Pixtral-12B-2409"
# MODEL_NAME = "google/gemma-2-9b-it"  # Text-only model

DATASET_NAME = "arampacha/rsicd"
MAX_LENGTH = 128
PROMPT_LENGTH = 4
BATCH_SIZE = 1
NUM_TRAIN_EPOCHS = 3
OUTPUT_DIR = "./manual_prompt_output"
device = "cuda" if torch.cuda.is_available() else "cpu"

## Universal VLM Prompt Tuning Implementation


In [ ]:
class UniversalVLMPromptTuning(nn.Module):
    def __init__(self, base_model, num_virtual_tokens=4):
        super().__init__()
        self.base_model = base_model
        self.num_virtual_tokens = num_virtual_tokens
        self.model_type = self._detect_model_type()

        # Get embedding dimension
        embed_dim = self._get_embedding_dim()

        # Create learnable soft prompts
        self.soft_prompts = nn.Parameter(
            torch.randn(num_virtual_tokens, embed_dim) * 0.1
        )

        # Initialize soft prompts
        nn.init.normal_(self.soft_prompts, std=0.02)

        # Freeze base model parameters
        self._freeze_base_model()

    def _detect_model_type(self):
        model_name = str(type(self.base_model).__name__).lower()
        config_type = str(type(self.base_model.config).__name__).lower()

        if "qwen" in model_name or "qwen" in config_type:
            return "qwen"
        elif "gemma" in model_name or "gemma" in config_type:
            return "gemma"
        elif "phi" in model_name or "phi" in config_type:
            return "phi"
        elif "pixtral" in model_name or "pixtral" in config_type:
            return "pixtral"
        return "unknown"

    def _get_embedding_dim(self):
        config = self.base_model.config

        # Try common embedding dimension attributes
        candidates = ['hidden_size', 'd_model', 'n_embed', 'dim']

        for attr in candidates:
            if hasattr(config, attr):
                return getattr(config, attr)

        # Check text_config for multimodal models
        if hasattr(config, 'text_config'):
            for attr in candidates:
                if hasattr(config.text_config, attr):
                    return getattr(config.text_config, attr)

        # Fallback: inspect embedding layer
        return self.base_model.get_input_embeddings().embedding_dim

    def _freeze_base_model(self):
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Only soft prompts are trainable
        self.soft_prompts.requires_grad = True

    def _prepare_inputs_with_prompts(self, input_ids, attention_mask):
        # Get text embeddings
        text_embeds = self.base_model.get_input_embeddings()(input_ids)
        batch_size = text_embeds.size(0)

        # Expand soft prompts for batch
        soft_prompts_batch = self.soft_prompts.unsqueeze(0).expand(
            batch_size, -1, -1
        )

        # Concatenate soft prompts with text embeddings
        combined_embeds = torch.cat([soft_prompts_batch, text_embeds], dim=1)

        # Extend attention mask for soft prompts
        prompt_mask = torch.ones(
            batch_size, self.num_virtual_tokens,
            device=attention_mask.device, dtype=attention_mask.dtype
        )
        extended_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        return combined_embeds, extended_mask

    def forward(self, input_ids, attention_mask, pixel_values=None, labels=None):
        # Prepare inputs with soft prompts
        combined_embeds, extended_mask = self._prepare_inputs_with_prompts(
            input_ids, attention_mask
        )

        # Prepare kwargs for model-specific forward pass
        kwargs = {
            'inputs_embeds': combined_embeds,
            'attention_mask': extended_mask,
            'labels': labels
        }

        # Add pixel_values for vision models (not for Gemma)
        if pixel_values is not None and self.model_type != "gemma":
            kwargs['pixel_values'] = pixel_values

        return self.base_model(**kwargs)

    def generate(self, input_ids, attention_mask, pixel_values=None, **generation_kwargs):
        # Prepare inputs with soft prompts
        combined_embeds, extended_mask = self._prepare_inputs_with_prompts(
            input_ids, attention_mask
        )

        # Prepare kwargs for generation
        kwargs = {
            'inputs_embeds': combined_embeds,
            'attention_mask': extended_mask,
            **generation_kwargs
        }

        # Add pixel_values for vision models
        if pixel_values is not None and self.model_type != "gemma":
            kwargs['pixel_values'] = pixel_values

        return self.base_model.generate(**kwargs)

    def print_trainable_parameters(self):
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

## Model Creation Helper


In [ ]:
def create_prompt_tuned_model(model_name, num_tokens=4):
    """Create a prompt-tuned model with universal VLM support"""
    print(f"Loading model: {model_name}")

    if "gemma" in model_name.lower():
        # Text-only model
        processor = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        # Vision-language model
        processor = AutoProcessor.from_pretrained(model_name, use_fast=True)
        model = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True
        )

    # Wrap with universal prompt tuning
    prompt_model = UniversalVLMPromptTuning(model, num_tokens)

    print(f"Model type detected: {prompt_model.model_type}")
    prompt_model.print_trainable_parameters()

    return processor, prompt_model

## Load Dataset


In [ ]:
dataset = load_dataset(DATASET_NAME)

# Dataset splits as planned
train_dataset = dataset["train"].select(range(1000))  # 1000 images for training
eval_dataset = dataset["valid"].select(range(200))    # 200 images for evaluation
test_dataset = dataset["test"].select(range(10))      # 10 images for testing

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Features: {train_dataset.features}")

## Load Model and Processor


In [ ]:
processor, model = create_prompt_tuned_model(MODEL_NAME, PROMPT_LENGTH)

# Alternative model options:
# processor, model = create_prompt_tuned_model("microsoft/Phi-3.5-vision-instruct", PROMPT_LENGTH)
# processor, model = create_prompt_tuned_model("mistralai/Pixtral-12B-2409", PROMPT_LENGTH)
# processor, model = create_prompt_tuned_model("google/gemma-2-9b-it", PROMPT_LENGTH)

## Preprocess Dataset


In [ ]:
def preprocess_for_vlm(example):
    """Preprocess for vision-language models"""
    image = example["image"]

    # Format as conversation
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": "Describe this satellite image."}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["captions"][0]}
            ]
        }
    ]

    # Apply chat template
    text = processor.apply_chat_template(messages, tokenize=False)

    # Process inputs
    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

    # For causal LM, labels are same as input_ids
    result = {
        "input_ids": inputs["input_ids"].squeeze(),
        "attention_mask": inputs["attention_mask"].squeeze(),
        "labels": inputs["input_ids"].squeeze()
    }

    # Add pixel_values if present
    if "pixel_values" in inputs:
        result["pixel_values"] = inputs["pixel_values"].squeeze()

    return result

def preprocess_for_text_only(example):
    """Preprocess for text-only models like Gemma"""
    # For text-only models, we can only use the caption
    prompt = "Describe this satellite image: "
    target = example["captions"][0]
    text = prompt + target

    inputs = processor(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

    return {
        "input_ids": inputs["input_ids"].squeeze(),
        "attention_mask": inputs["attention_mask"].squeeze(),
        "labels": inputs["input_ids"].squeeze()
    }

# Choose preprocessing based on model type
if model.model_type == "gemma":
    preprocess_func = preprocess_for_text_only
else:
    preprocess_func = preprocess_for_vlm

# Apply preprocessing
train_dataset = train_dataset.map(preprocess_func)
eval_dataset = eval_dataset.map(preprocess_func)
test_dataset = test_dataset.map(preprocess_func)

train_dataset.set_format(type="torch")
eval_dataset.set_format(type="torch")
test_dataset.set_format(type="torch")

## Training Arguments


In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    logging_steps=10,
    eval_steps=100,
    save_steps=200,
    save_total_limit=1,
    evaluation_strategy="steps",
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    learning_rate=3e-2,  # Higher learning rate for prompt tuning
    warmup_steps=50
)

## Evaluation Metrics


In [ ]:
# Initialize evaluation metrics
cider_scorer = Cider()
spice_scorer = Spice()

def compute_cosine_similarity(predictions, references):
    """Compute cosine similarity between predictions and references"""
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizer = TfidfVectorizer()
    all_texts = predictions + [ref[0] for ref in references]

    try:
        tfidf_matrix = vectorizer.fit_transform(all_texts)
        pred_vectors = tfidf_matrix[:len(predictions)]
        ref_vectors = tfidf_matrix[len(predictions):]

        similarities = []
        for i in range(len(predictions)):
            sim = cosine_similarity(pred_vectors[i], ref_vectors[i])[0][0]
            similarities.append(sim)

        return np.mean(similarities)
    except:
        return 0.0

def compute_all_metrics(predictions, references):
    """Compute evaluation metrics using pycocoevalcap"""
    # Format data for pycocoevalcap (requires dict format)
    gts = {}  # ground truth
    res = {}  # results

    for i, (pred, ref_list) in enumerate(zip(predictions, references)):
        gts[i] = ref_list
        res[i] = [pred]

    # Compute metrics
    results = {}

    try:
        # CIDEr (TF-IDF + cosine)
        cider_score, _ = cider_scorer.compute_score(gts, res)
        results['CIDEr'] = cider_score
    except Exception as e:
        print(f"CIDEr calculation failed: {e}")
        results['CIDEr'] = 0.0

    try:
        # SPICE (F1 with scene graphs)
        spice_score, _ = spice_scorer.compute_score(gts, res)
        results['SPICE'] = spice_score
    except Exception as e:
        print(f"SPICE calculation failed: {e}")
        results['SPICE'] = 0.0

    # Cosine similarity
    results['Cosine_Similarity'] = compute_cosine_similarity(predictions, references)

    return results

def compute_metrics(eval_preds):
    """Compute metrics during training"""
    logits, labels = eval_preds
    predictions = torch.argmax(torch.tensor(logits), dim=-1)

    # Decode predictions and labels
    decoded_preds = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Clean predictions (extract generated part)
    cleaned_preds = []
    for pred in decoded_preds:
        if "assistant" in pred:
            cleaned_pred = pred.split("assistant")[-1].strip()
        else:
            cleaned_pred = pred.strip()
        cleaned_preds.append(cleaned_pred)

    # Compute subset of metrics for training (faster)
    try:
        gts = {i: [ref] for i, ref in enumerate(decoded_labels)}
        res = {i: [pred] for i, pred in enumerate(cleaned_preds)}

        cider_score, _ = cider_scorer.compute_score(gts, res)
        cosine_sim = compute_cosine_similarity(cleaned_preds, [[l] for l in decoded_labels])

        return {
            "CIDEr": cider_score,
            "Cosine_Similarity": cosine_sim
        }
    except:
        return {
            "CIDEr": 0.0,
            "Cosine_Similarity": 0.0
        }

## Train the Model


In [ ]:
# Custom trainer for manual prompt tuning
class PromptTuningTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Forward pass through our custom model
        outputs = model(**inputs)
        loss = outputs.loss
        return (loss, outputs) if return_outputs else loss

# Move model to device before creating trainer (fixes meta tensor issue)
model = model.to(device)

trainer = PromptTuningTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor.tokenizer if hasattr(processor, 'tokenizer') else processor,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

## Test on Multiple Samples


In [ ]:
# Test on the 10 test samples
predictions = []
references = []
inference_times = []
vram_usage = []

# Get original test dataset for references
original_test = dataset["test"].select(range(10))

for i in range(len(original_test)):
    original_sample = original_test[i]

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    # Prepare inference input
    if model.model_type == "gemma":
        # Text-only inference
        prompt = "Describe this satellite image: "
        inputs = processor(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=64,
                temperature=0.7,
                do_sample=True
            )
    else:
        # Vision-language inference
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": original_sample["image"]},
                    {"type": "text", "text": "Describe this satellite image."}
                ]
            }
        ]

        input_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=input_text, images=original_sample["image"], return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pixel_values=inputs.get("pixel_values"),
                max_new_tokens=64,
                temperature=0.7,
                do_sample=True
            )

    end_time = time.time()

    # Decode output
    generated_text = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract generated part
    if "assistant" in generated_text:
        generated_part = generated_text.split("assistant")[-1].strip()
    else:
        generated_part = generated_text.strip()

    predictions.append(generated_part)
    references.append([original_sample["captions"][0]])
    inference_times.append(end_time - start_time)
    vram_usage.append(torch.cuda.max_memory_allocated() / 1e9)

    print(f"Sample {i+1}:")
    print(f"Generated: {generated_part}")
    print(f"Reference: {original_sample['captions'][0]}")
    print(f"Inference Time: {inference_times[-1]:.3f}s")
    print(f"Peak VRAM: {vram_usage[-1]:.2f}GB")
    print("-" * 50)

# Calculate comprehensive metrics using pycocoevalcap
print("\n=== Final Test Results ===")
print(f"Average Inference Time: {np.mean(inference_times):.3f}s (±{np.std(inference_times):.3f}s)")
print(f"Average VRAM Usage: {np.mean(vram_usage):.2f}GB (±{np.std(vram_usage):.2f}GB)")

print("\n=== Evaluation Metrics ===")
metrics = compute_all_metrics(predictions, references)
for metric_name, score in metrics.items():
    print(f"{metric_name}: {score:.4f}")

# Print detailed results table
print("\n=== Metrics Summary ===")
print(f"{'Metric':<15} {'Score':<10}")
print("-" * 25)
for metric_name, score in metrics.items():
    print(f"{metric_name:<15} {score:<10.4f}")

## Save Model


In [ ]:
# Save the soft prompts
torch.save({
    'soft_prompts': model.soft_prompts,
    'model_type': model.model_type,
    'num_virtual_tokens': model.num_virtual_tokens,
    'model_name': MODEL_NAME
}, f"{OUTPUT_DIR}/soft_prompts.pt")

print(f"Soft prompts saved to {OUTPUT_DIR}/soft_prompts.pt")
print(f"Model type: {model.model_type}")
print(f"Virtual tokens: {model.num_virtual_tokens}")

## Load and Test Saved Model


In [ ]:
# Example of loading saved soft prompts
def load_soft_prompts(checkpoint_path, base_model):
    checkpoint = torch.load(checkpoint_path)

    # Create new prompt tuning model
    prompt_model = UniversalVLMPromptTuning(
        base_model,
        num_virtual_tokens=checkpoint['num_virtual_tokens']
    )

    # Load the trained soft prompts
    prompt_model.soft_prompts.data = checkpoint['soft_prompts']

    print(f"Loaded soft prompts for {checkpoint['model_type']} model")
    return prompt_model

# To load later:
# processor, base_model = create_prompt_tuned_model(MODEL_NAME, PROMPT_LENGTH)
# loaded_model = load_soft_prompts(f"{OUTPUT_DIR}/soft_prompts.pt", base_model.base_model)